# AF2 radial-wavelet refinement — sequential Kaggle
Runs `AF2RAD → AF2WAV → AF2RADWAV` on seed 42 under the frozen follow-up protocol. Attach only the private `faruq-v3-experiment-core-v1` dataset. Optionally attach a prior Saved Version of this same notebook for resume. Test remains locked. At successful completion the notebook keeps a small handoff instead of the full extracted dataset/checkpoint tree.


In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini Kaggle-only.')
manifests=sorted(p for p in INPUT.rglob('af2_spectral_kaggle_manifest.json') if p.is_file())
d0=sorted(p for p in INPUT.rglob('D0_seed42_best.pt') if p.is_file())
if len(manifests)!=1 or len(d0)!=1: raise FileNotFoundError(f'STOP CEPAT: manifest={manifests}, D0={d0}')
print('FAST INPUT PREFLIGHT PASS'); print('MANIFEST:',manifests[0]); print('D0:',d0[0])


In [ ]:
import importlib,json,os,shutil,subprocess,sys,time
from pathlib import Path
WORK=Path('/kaggle/working'); INPUT=Path('/kaggle/input'); REPO=WORK/'coffee-bean-detection'; OUT=WORK/'af2-rad-wavelet-refinement-v1'; BRANCH='agent/af2-rad-wavelet-refinement'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip(); print('COMMIT:',COMMIT)
from coffee_detector.af2_refinement import TRAIN_ARMS,run_af2_refinement_static_audit
from coffee_detector.experiments.prepare_af2_refinement_kaggle import prepare_af2_refinement_kaggle_input,restore_af2_refinement_run
from coffee_detector.experiments.run_faruq_v3_af2_refinement_decision import run_af2_refinement_decision
DATA,A,CONTRACT=prepare_af2_refinement_kaggle_input(INPUT,WORK); assert CONTRACT['decision']=='PASS' and CONTRACT['test_images_accessed'] is False
D0=A['D0_seed42_best.pt']; BASELINE=A['lfdet_afab_seed42_screening.json']; OUT.mkdir(exist_ok=True); STATIC=OUT/'static_audit.json'
audit=run_af2_refinement_static_audit(D0,STATIC,device='cuda:0')
if audit['decision']!='PASS': raise RuntimeError(f'STOP: static audit gagal: {audit}')
assert audit['training_authorized'] is True and audit['test_access_authorized'] is False
print('STATIC AUDIT PASS')

def run_arm(arm):
    config=REPO/f'configs/af2_refinement/{arm}_yolo26n.yaml'
    restored=restore_af2_refinement_run(INPUT,OUT,arm=arm,seed=42,d0_checkpoint=D0,config=config)
    result=OUT/'val_reports'/f'{arm}_seed42_result.json'; log=OUT/f'{arm}_seed42_run.log'
    command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_refinement_arm','--arm',arm,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--d0-checkpoint',str(D0),'--static-audit',str(STATIC),'--output-root',str(OUT),'--seed','42','--device','0','--authorize-training']
    print(f'START {arm} | restored={restored}',flush=True)
    with log.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    seen=None
    while process.poll() is None:
        csv=OUT/arm/f'{arm}_seed42'/'results.csv'; epoch=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epoch!=seen: print(f'{arm}: {epoch}/50 epoch | log={log}',flush=True); seen=epoch
        time.sleep(120)
    if process.returncode:
        tail='\n'.join(log.read_text(errors='replace').splitlines()[-180:]) if log.is_file() else '<log tidak ditemukan>'
        raise RuntimeError(f'{arm} gagal: returncode={process.returncode}\n--- LOG TAIL ---\n{tail}')
    if not result.is_file(): raise FileNotFoundError(result)
    payload=json.loads(result.read_text(encoding='utf-8')); assert payload['evaluation_split']=='val' and payload['test_images_accessed'] is False
    m=payload['metrics']; print(arm,{'Macro':m['macro_map50_95'],'Bottom3':m['bottom3_class_map50_95'],'Worst':m['worst_class_map50_95'],'latency_ms':payload['latency']['median_ms']},flush=True)

for arm in TRAIN_ARMS: run_arm(arm)
decision=run_af2_refinement_decision(OUT,BASELINE,seed=42); assert decision['test_opened'] is False
print('=== FINAL SEED42 DECISION ==='); print(json.dumps(decision,indent=2))

# Small Saved-Version handoff: keep evidence, not the extracted 1.1GB dataset or all checkpoints.
HANDOFF=WORK/'af2-rad-wavelet-refinement-handoff'; shutil.rmtree(HANDOFF,ignore_errors=True); (HANDOFF/'val_reports').mkdir(parents=True)
for arm in TRAIN_ARMS:
    for suffix in ('result.json','latency.json','val.json'):
        src=OUT/'val_reports'/f'{arm}_seed42_{suffix}'
        if src.is_file(): shutil.copy2(src,HANDOFF/'val_reports'/src.name)
for src in (STATIC,OUT/'val_reports/af2_refinement_seed42_decision.json'):
    if src.is_file(): shutil.copy2(src,HANDOFF/src.name)
winner=decision.get('winner')
if winner:
    best=OUT/winner/f'{winner}_seed42'/'weights/best.pt'
    if best.is_file(): shutil.copy2(best,HANDOFF/f'{winner}_seed42_best.pt')
manifest={'format':'coffee_detector.af2_refinement.handoff.v1','commit':COMMIT,'seed':42,'decision':decision['decision'],'winner':winner,'retained':decision['retained'],'test_images_accessed':False}
(HANDOFF/'handoff_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n',encoding='utf-8')
# Cleanup only after complete decision/handoff has been materialized.
if DATA.exists(): shutil.rmtree(DATA)
for arm in TRAIN_ARMS: shutil.rmtree(OUT/arm,ignore_errors=True)
if REPO.exists(): shutil.rmtree(REPO)
print('HANDOFF READY:',HANDOFF); print('SAVE VERSION. Tidak perlu download checkpoint.')
